#AutoML(Automated Machine Learning)

사용자가 직접 모델을 설계·튜닝하지 않아도, 데이터 전처리 → 모델 선택 → 하이퍼파라미터 최적화까지 **자동으로 수행**하는 기술.

| 단계 | 설명 | 예시 |
|------|------|------|
| 데이터 전처리 | 결측치 처리, 스케일링, 인코딩 | `StandardScaler`, `OneHotEncoder` |
| 모델 탐색 | 여러 알고리즘 비교·평가 | `RandomForest`, `XGBoost`, `LightGBM`, `CatBoost`, ... |
| 하이퍼파라미터 탐색 | 각 모델의 설정값 자동 최적화 | `max_depth`, `learning_rate`, `n_estimators` 등 |
| 평가 및 선택 | 교차검증 성능 비교 후 최적 모델 선택 | RMSE, MAE, R² 등 |

---

- 머신러닝 모델의 성능은 **하이퍼파라미터 설정에 매우 민감**하다.  
- 수작업으로 모든 조합을 시도하기엔 **시간·자원 낭비**가 크다.  
- AutoML은 **탐색과 최적화 과정을 자동화**하여 연구자나 실무자가 **데이터 해석과 의사결정**에 집중할 수 있도록 돕는다.


# iris_data를 xgboost로 적용을 해보자.

In [ ]:
#import packages
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris

#Loading iris dataset from sklearn
iris = load_iris()

#independent feautres
X = iris.data

# target features
y = iris.target

In [ ]:
#import XGboost
from xgboost import XGBClassifier

#Defining XGB Classification model
clf = XGBClassifier()

- xgboost의 단점은 복잡한 하이퍼 파라미터가 있다.

# 1.Grid SearchCV

---

사용자가 지정한 하이퍼파라미터 후보값들을 **모든 조합으로 탐색(Grid Search)** 하여,  가장 좋은 성능을 내는 파라미터를 찾는 방법이다.

즉, 각 하이퍼파라미터에 대해 가능한 값을 리스트로 입력하면, 모든 가능한 조합을 반복적으로 학습·평가하여 **최적의 하이퍼파라미터 조합**을 자동으로 찾아준다.

---

**주요 특징**

| 항목 | 설명 |
|------|------|
| **탐색 방식** | 모든 하이퍼파라미터 조합을 완전 탐색 |
| **평가 방법** | 교차검증(Cross Validation)을 이용해 각 조합의 평균 성능 계산 |
| **장점** | 단순하고 구현이 쉬움 |
| **단점** | 조합 수가 많아질수록 계산 비용이 급격히 증가 |

In [ ]:
#Importing packages from sklearn

from sklearn import preprocessing
from sklearn import model_selection
from sklearn import metrics

#defining a set of values as a dictionary for hyperparameters

param_grid = {
    "n_estimators":[100,200,300,400],
    "max_depth":[1,3,5,7],
    "reg_lambda":[.01,.1,.5]
}

#declaring GridSearchCV model

model = model_selection.GridSearchCV(
    estimator = clf,
    param_grid = param_grid,
    scoring = 'accuracy',
    verbose = 10,
    n_jobs = 1,
    cv = 5
)

#fitting values to the gridsearchcv model

model.fit(X,y)
#printing the best possible values to enhance accuracy
print(model.best_params_)
print(model.best_estimator_)
#printing the best score
print(model.best_score_)
#GridSearchCV model save
# import joblib
# joblib.dump(model, 'model_xgboost.pkl')
# joblib.load('model_xgboost.pkl')

# 2.RandomizedSearchCV

---
그리드 서치처럼 모든 조합을 탐색하지 않고, **지정된 하이퍼파라미터 공간에서 무작위로 일부 조합만 선택**하여 탐색하는 방식이다.

하이퍼파라미터 후보 범위가 넓거나 연속적인 경우, 모든 경우를 탐색하기 어려울 때 효율적으로 **탐색 시간과 자원**을 절약할 수 있다.

---

**주요 특징**

| 항목 | 설명 |
|------|------|
| **탐색 방식** | 지정한 횟수(`n_iter`)만큼 무작위 샘플링 |
| **평가 방법** | 교차검증(Cross Validation)으로 성능 평가 |
| **장점** | 연산 비용 절감, 고차원 탐색에 유리 |
| **단점** | 완전 탐색이 아니므로 최적값을 놓칠 가능성 존재 |


In [ ]:
#defining a set of values as a dictionary for hyperparameters

param_grid = {
    "n_estimators":[100,200,300,400],
    "max_depth":[1,3,5,7],
    "reg_lambda":[.01,.1,.5]
}

#declaring RandomizedSearchCV model

model = model_selection.RandomizedSearchCV(
    estimator = clf,
    param_distributions = param_grid,
    scoring = 'accuracy',
    verbose = 10,
    n_jobs = 1,
    cv = 5,
    n_iter=10
)

#fitting values to the RandomizedSearchCV model

model.fit(X,y)

#printing the best possible values to enhance accuracy

print(model.best_params_)
print(model.best_estimator_)
#printing the best score
print(model.best_score_)

- 분포를 활용한 RandomSerachCV

In [ ]:
import xgboost as xgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import model_selection, metrics
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from scipy.stats import randint, uniform, loguniform  # 분포

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

param_dist = {
    "n_estimators": randint(300, 1500),        # 트리 개수
    "max_depth": randint(2, 8),                # 트리 깊이
    "min_child_weight": randint(1, 10),        # 리프 최소 가중치합
    "subsample": uniform(0.6, 0.4),            # 표본 샘플 비율 [0.6, 1.0)
    "colsample_bytree": uniform(0.6, 0.4),     # 특성 샘플 비율 [0.6, 1.0)
    "gamma": loguniform(1e-3, 1.0),            # 분할 최소 손실 감소
    "reg_lambda": loguniform(1e-2, 10),        # L2
    "reg_alpha": loguniform(1e-3, 1),          # L1
    "learning_rate": loguniform(5e-2, 2e-1),   # 0.05~0.2 근방
}

clf = xgb.XGBClassifier(
    tree_method="hist",
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

rs = model_selection.RandomizedSearchCV(
    estimator=clf,
    param_distributions=param_dist,
    n_iter=40,                 # 예: 40회 시도 (시간에 맞게 조절)
    scoring="accuracy",        # 요청과 동일하게 accuracy 사용
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42,
)


rs.fit(X_train, y_train)

print("Best Params:", rs.best_params_)
print("CV Best Score (accuracy):", round(rs.best_score_, 4))
best_model = rs.best_estimator_


y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]
print("Test Accuracy:", round(metrics.accuracy_score(y_test, y_pred), 4))
print("Test ROC-AUC :", round(metrics.roc_auc_score(y_test, y_proba), 4))

imp = best_model.feature_importances_
order = np.argsort(imp)
plt.figure(figsize=(7, 8))
plt.barh(np.array(X.columns)[order], imp[order])
plt.title("XGBoost Feature Importance")
plt.tight_layout(); plt.show()

# 3. Bayesian Optimization

---
  
입력값 $ x $에 대한 미지의 목적 함수 $~f(x)~$를 탐색하여 그 함숫값 $~f(x)~$를 **최대화(또는 최소화)하는 최적해**를 찾는 방법이다.

이 방법은 목적 함수를 직접 계산하기 어렵거나 시간이 많이 드는 경우 (예: 딥러닝 모델의 하이퍼파라미터 탐색)에 매우 효과적이다.  

목적 함수를 **black-box function**이라 부르며, 이를 근사하기 위해 확률적 모델(Surrogate Model)을 사용한다.

---

**1) Surrogate Model (대체 모델)**

실제 목적함수 $f(x) $를 직접 계산하지 않고, 현재까지의 관측 $ D_t=\{(x_1,f(x_1)),\dots,(x_t,f(x_t))\} $로 **확률적 근사**를 만든다.  


이 근사 모델이 **각 후보 $x$에서의 예측 평균 $\mu(x)$와 불확실성 $\sigma(x)$를 제공**.

- $f(x)$ 평가가 고비용일 때, 싸고 빠른 근사로 "어디를 다음에 평가할지"결정하기 위함.

- 불확실성 정보를 함께 제공해 탐색(Exploration)을 정량화.

---

| 모델 | 핵심 아이디어 | 장점 | 단점/주의 |
|---|---|---|---|
| **Gaussian Process (GP)** | 모든 점의 $f(x)$가 다변량 정규. 커널 $k(x,x')$로 상관 구조 모델링 | $\mu,\sigma$를 폐형식으로 제공. 불확실성 정량화 우수 | 고차원·대규모 데이터에서 비효율, 커널·하이퍼파라미터 민감 |
| **TPE (Tree-structured Parzen Estimator)** | “좋은 성능”과 “나쁜 성능” 영역의 파라미터 분포를 각각 KDE로 추정 | 범주·연속 혼합, 고차원에도 실용적 | 이론적 해석은 GP보다 덜 투명 |
        "폐형식(closed form)"은 수학·통계에서 명시적인 계산식으로 표현 가능한 형태. 어떤 값을 구하기 위해 반복적 수치계산이나 근사법이 필요 없는 형태를 말함.
        가우시안 프로세스(GP)는 예측값의 평균과 불확실성 을미리 정해진 수식으로 직접 계산할 수 있어서“폐형식으로 계산된다”고 함.

---
### **Gaussian Process (GP)**

입력과 출력 사이의 관계를 커널 함수 $k(x, x')$ 로 표현하여  새로운 입력값에 대한 예측 평균($\mu$)과 불확실성(분산 $\sigma^2$)을 계산하는 모델입니다.

---


- 입력 데이터 : $~X = [x_1, x_2, \dots, x_t] $
- 출력 데이터 : $~y = [f(x_1), f(x_2), \dots, f(x_t)]^\top$
- 새 입력점 : $~x_* $

가우시안 프로세스는 모든 함수값이 다변량 정규분포를 따른다고 가정합니다.

$$
\begin{bmatrix}
y \\ f(x_*)
\end{bmatrix}
\sim
\mathcal{N}\left(
0,
\begin{bmatrix}
K(X,X)+\sigma_n^2 I & K(X,x_*)\\
K(x_*,X) & K(x_*,x_*)
\end{bmatrix}
\right)
$$

여기서  
- $K(X,X)$ : 훈련 데이터 간 커널 행렬  
- $K(X,x_*)$ : 훈련 데이터와 새 입력 간의 커널 벡터  
- $\sigma_n^2 I$ : 관측 노이즈(측정 오차)

---

###**커널 행렬 (Kernel Matrix)**

입력 데이터 간의 유사도(similarity)를 수학적으로 표현한 행렬

가우시안 프로세스(GP)에서는 이 행렬이 공분산 행렬(covariance matrix)의 역할을 하며, 각 입력 쌍 $(x_i, x_j)$에 대해 커널 함수 $k(x_i, x_j)$를 계산하여 구성

---

훈련 데이터가 $ X = [x_1, x_2, ..., x_t] $일 때, 커널 행렬 $ K(X, X) $는 다음과 같이 정의됩니다.

$$
K(X,X) =
\begin{bmatrix}
k(x_1,x_1) & k(x_1,x_2) & \dots & k(x_1,x_t) \\
k(x_2,x_1) & k(x_2,x_2) & \dots & k(x_2,x_t) \\
\vdots & \vdots & \ddots & \vdots \\
k(x_t,x_1) & k(x_t,x_2) & \dots & k(x_t,x_t)
\end{bmatrix}
$$



행 $i$, 열 $j$의 원소는 두 데이터 $x_i$와 $x_j$ 사이의 유사도를 나타냅니다.

---


| 커널 이름 | 수식 | 의미 |
|------------|------|------|
| **RBF (Radial Basis Function)** | $$k(x_i,x_j)=\exp\!\left(-\frac{\|x_i-x_j\|^2}{2l^2}\right)$$ | 거리가 가까울수록 높은 유사도 |
| **Linear Kernel** | $$k(x_i,x_j)=x_i^\top x_j$$ | 선형 관계 표현 |
| **Polynomial Kernel** | $$k(x_i,x_j)=(x_i^\top x_j + c)^d$$ | 다항식 형태의 비선형 관계 |
| **Matern Kernel** | $$k(r)=\frac{1}{\Gamma(\nu)2^{\nu-1}}\!\left(\frac{\sqrt{2\nu}r}{l}\right)^\nu K_\nu\!\left(\frac{\sqrt{2\nu}r}{l}\right)$$ | 함수의 부드러움을 조절 가능 |

---



**2) Acquisition Function (획득 함수)**

Surrogate Model의 $\mu(x),\sigma(x)$를 이용해 다음 평가점 $x_{t+1}$을 고르는 기준 함수 $a(x)$. 핵심은 **탐색–활용 균형**을 수식으로 구현하는 것.

$$
x_{t+1}=\arg\max_x a(x \mid D_t)
$$


**대표 획득 함수**

**(a) Expected Improvement (EI)**

현재 최적값 $f^{+}=\max_{i\le t} f(x_i)$ 대비 기대 개선량의 기댓값.

$$
\text{EI}(x)=\mathbb{E}[(f(x)-f^{+})_+]
$$

GP 가정에서
$$
z=\frac{\mu(x)-f^{+}-\xi}{\sigma(x)},\quad
\text{EI}(x)=(\mu-f^{+}-\xi)\Phi(z)+\sigma\phi(z)
$$
- $\Phi,\phi$: 표준정규 CDF, PDF  
- $\xi\ge0$: 탐색 강화 상수. $\xi$가 크면 탐색 성향 증가.

**특징**: 개선의 “크기”와 “확률”을 동시에 고려. 실무 기본값으로 널리 사용.

**(b) Probability of Improvement (PI)**
개선이 일어날 **확률**만 최대로.

$$
\text{PI}(x)=\mathbb{P}(f(x)>f^{+}+\xi)=\Phi\!\left(\frac{\mu(x)-f^{+}-\xi}{\sigma(x)}\right)
$$
**특징**: 단순하고 빠름. 개선 “크기” 무시 → 미세개선에 집착 위험.

**(c) Upper Confidence Bound (UCB)**
평균과 불확실성의 가중 합.
$$
\text{UCB}(x)=\mu(x)+\kappa\,\sigma(x)
$$
- $\kappa>0$: 탐색 강도. 클수록 미탐색 영역 선호.  
**특징**: 구현 간단, 해석 직관적. \(\kappa\) 튜닝이 성능 좌우.


- **EI**: 안정적 기본 선택. 개선 크기까지 고려.  
- **UCB**: 제어 가능한 탐색–활용 스위치가 필요할 때.  
- **PI**: 계산 단순·고속이 중요할 때나 사전 탐색 단계.

<img src='https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdn%2Fb33tsP%2FbtraMpvxJG0%2FSn7uQK7k910IQ7cP3ZM9vk%2Fimg.png'>

위 그림은 **Bayesian Optimization**의 전체 탐색 과정을 시각화한 것입니다.   상단 그래프는 목적함수 $f(x)$와 그에 대한 예측 모델(Surrogate Model)을 보여주며, 하단 그래프는 획득함수(Acquisition Function, EI)를 통해 다음 실험 위치를 결정하는 과정을 나타냄.

---
상단 그래프 (Surrogate Model)

| 요소 | 설명 |
|------|------|
| **파란색 실선** | 우리가 찾고자 하는 실제 **목적함수 f(x)** |
| **빨간색 점** | 현재까지 관측된 입력값–출력값 데이터 포인트 |
| **검정색 점선** | 관측 데이터를 기반으로 Surrogate Model이 예측한 **평균 함수(Estimated Function)** |
| **파란 음영 영역** | Surrogate Model이 예측한 함수의 **불확실성(Variance, Confidence Interval)** |
| **넓은 음영** | 아직 탐색이 부족하여 불확실성이 큰 구간 |
| **좁은 음영** | 이미 많이 탐색되어 불확실성이 작은 구간 |

즉, Surrogate Model은 실제 함수를 근사하며, 탐색이 진행될수록 예측 곡선(검정 점선)이 실제 함수(파란 실선)에 가까워짐.

---
(Acquisition Function)

| 요소 | 설명 |
|------|------|
| **보라색 곡선 (EI(x))** | Expected Improvement: 개선 가능성이 큰 구간을 나타내는 **획득 함수** |
| **노란 별 표시** | EI(x)가 가장 큰 지점 → **다음 입력값 후보 (Next Candidate)** |

EI(x)가 높은 구간은 "현재까지의 정보로 볼 때 성능이 더 좋아질 가능성이 큰" 구간입니다.  Bayesian Optimization은 이 구간의 입력값을 **다음 실험점으로 선택**



- 직관적인 이해

In [ ]:
# - 목표: f(x)를 최대화할 x 찾기
# - Surrogate: Gaussian Process
# - Acquisition: Expected Improvement(EI)
# - 의존성: scikit-learn, numpy, matplotlib

# 목표 : f(x)를 최대화할 x를 찾기
## - surrogate : Gaussian process
## - Acquistion : Expected Improvement(EI)
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel

#목적함수 정의
def f(x):
    x = np.asarray(x)
    return (np.sin(12*x) + 0.6 * np.cos(5*x) + 0.1*np.sin(25*x)) * (0.5 + 4*x)

#탐색 구간 설정.
bounds = (0.0, 1.0)

# 초기 관측 n개 샘플
rng = np.random.RandomState(42)
n_init = 4
X = rng.uniform(bounds[0],bounds[1],size = (n_init,1))
y = f(X).ravel()

#EI 함수 구현
from scipy.stats import norm

def expected_improvement(Xcand, model, y_best,xi=1e-3):
    '''
    베이지안 최적화에서 탐색 후보 지점(Xcand)마다 기대개선값(Expected_improvement)을 계산하여
    다음 샘플링 지점을 선택하는 기준으로 사용

    <매개변수>
    - Xcand : ndarray of shape (m,d) / 후보 입력점(candidate points)
    - model : GaussianProcessRegressor / 예측 모델(보통 Gaussian Process)
    - y_best : float / 현재까지의 최고 관측값
    - xi : float / 탐색(exploration) VS 활용(exploitation) 균형 조절 상수
    - Return : ei(ndarray of shape (m,) /각 후보점에서의 Expected Improvement)
    '''
    # 후보점에서 예측평균과 표준편차 계산
    mu, sigma = model.predict(Xcand, return_std=True)
    #수치 안정화 확보 : sigma=0을 방지 (분모가 0이 되는 것을 방지) -> NaN을 방지
    sigma = np.maximum(sigma,1e-9) # 수치 안정화
    #계산량(imp) : 예측값이 현재 최고값을 얼마나 넘는가?
    imp = mu - y_best - xi
    #표준화된 계산량 (+예측 불확실성을 반영하여 정규화)
    Z = imp / sigma
    # EI식을 그대로 사용한것이고 cdf(표준정규 누적분포함수) / pdf(표준정규 확률밀도함수)
    ei = imp * norm.cdf(Z)+ sigma * norm.pdf(Z)
    #예측 불확실성이 거의 없는 지점(sigma=0)은 Ei=0을 설정
    ei[sigma < 1e-9] = 0.0
    return ei

# 5) BO 루프
n_iter = 300
for t in range(n_iter):
    # GP 적합
    kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(length_scale=0.02, length_scale_bounds=(1e-3, 1.0)) \
             + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))
    gp = GaussianProcessRegressor(kernel=kernel, alpha=0.0, normalize_y=True, random_state=0)
    gp.fit(X, y)

    # 후보 격자에서 EI 최대점 선택
    X_grid = np.linspace(bounds[0], bounds[1], 600).reshape(-1, 1)
    ei = expected_improvement(X_grid, gp, y_best = float(np.max(y)), xi=1e-3)
    x_next = X_grid[np.argmax(ei)].reshape(1, 1)

    # 새 점 평가 후 데이터 갱신
    y_next = f(x_next).ravel()
    X = np.vstack([X, x_next])
    y = np.hstack([y, y_next])

# 6) 최종 시각화
gp.fit(X, y)
X_plot = np.linspace(bounds[0], bounds[1], 600).reshape(-1, 1)
mu, std = gp.predict(X_plot, return_std=True)

fig = plt.figure(figsize=(10, 6))

# 상단: Surrogate 예측과 불확실성
ax1 = plt.subplot2grid((3, 1), (0, 0), rowspan=2)
ax1.plot(X_plot, f(X_plot), label="Objective f(x)", linewidth=2)
ax1.plot(X_plot, mu, "k--", label="GP Prediction", linewidth=2)
ax1.fill_between(X_plot.ravel(), mu-2*std, mu+2*std, alpha=0.25, label="Uncertainty (±2σ)")
ax1.scatter(X[:-1], y[:-1], color="crimson", edgecolor="k", zorder=5, label="Observations", s=50)
ax1.scatter(X[-1], y[-1], color="gold", edgecolor="k", zorder=6, label="Last eval", s=70)
ax1.set_ylabel("f(x)")
ax1.set_xlim(bounds)
ax1.legend(loc="upper right", frameon=False)
ax1.set_title("Bayesian Optimization with GP + Expected Improvement")

# 하단: EI와 다음 후보
ei = expected_improvement(X_plot, gp, y_best = float(np.max(y)), xi=1e-3)
x_star = X_plot[np.argmax(ei)]
ax2 = plt.subplot2grid((3, 1), (2, 0))
ax2.plot(X_plot, ei, label="EI(x)")
ax2.scatter([x_star], [ei.max()], color="gold", edgecolor="k", zorder=6, label="Next candidate", s=70)
ax2.set_xlim(bounds)
ax2.set_xlabel("x")
ax2.set_ylabel("EI(x)")
ax2.legend(loc="upper right", frameon=False)

plt.tight_layout()
plt.show()

# 7) 결과 출력
x_opt_idx = np.argmax(y)
print("관측 중 최고값 x* =", float(X[x_opt_idx].item()), " f(x*) =", float(y[x_opt_idx].item()))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
#확률적 회귀 모델, 데이터 포인트 사이의 관계를 확률분포로 모델링 하는 것.
# 특징 : 비선형 관계를 학습할 수 있음. 샘플 개수가 적어도 신뢰성 있는 예측할수 있다.
# 특징 : 불확실성 정량적으로 평가 가능. / 커널 : 데이터 포인트 사이의 유사도를 측정하는 함수.
# 가우시안 프로세스(GP)에서 커널은 "이 점과 저 점이 얼마나 비슷한가?"
#RBF(Radial Basis Function,aka Gaussian kernel)
# 가장 부드러운 커널. 모든 점이 서로 영향을 미칠 가능성이 있음.
# Matern 커널 : RBF보다는 매끄럽지는 않음. 함수의 거칠기(roughtness)조절할 수 있음.
# -> RBF : 매우 부드러운 변화를 가짐. / Matern : 덜 부드럽고 급격한 변화도 허용.
from sklearn.gaussian_process.kernels import Matern, RBF

# 목적 함수 (최적화할 함수)
# 주어진 x 값에 대해 비선형 함수의 값을 반환
# 이 함수는 최적화를 수행할 대상이며, 샘플링을 통해 최적값을 찾아간다.
def objective_function(x):
    return np.sin(3*x) + x**2 - 0.7*x

# 획득 함수 (Expected Improvement, EI)
# 현재까지의 샘플링 데이터를 기반으로 획득 함수 값을 계산하여, 다음 샘플링할 최적의 지점을 결정한다.
def expected_improvement(X, X_sample, Y_sample, gpr, xi=0.01):
    mu, sigma = gpr.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)  # 2D 형태 유지
    mu_sample = gpr.predict(X_sample)
    mu_sample_opt = np.max(mu_sample)

    # np.errstate를 사용하여 경고 메시지를 제어
    # divide='warn': 0으로 나누는 연산이 발생할 경우 경고를 발생시키고 오류를 방지
    with np.errstate(divide='warn'):
        improvement = mu - mu_sample_opt - xi
        Z = improvement / sigma  # sigma가 0일 경우 문제가 발생할 수 있음
        ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

        ei = ei.flatten()  # 1D 배열로 변환
        sigma = sigma.flatten()  # 1D 배열로 변환

        if ei.shape == sigma.shape:
            ei[sigma == 0.0] = 0.0  # 0인 값 처리

    return ei

# 베이지안 최적화 실행 함수
# 초기 샘플링 데이터를 기반으로 가우시안 프로세스를 학습하며, 최적의 지점을 찾아간다.
def bayesian_optimization(n_iterations=10, n_initial=3, kernel_type='Matern'):
    np.random.seed(42)

    # 검색 범위
    X = np.linspace(-1, 2, 100).reshape(-1, 1)

    # 초기 샘플링 데이터
    X_sample = np.random.uniform(-1, 2, size=(n_initial, 1))
    Y_sample = objective_function(X_sample)

    # 커널 선택 (Matern 또는 RBF)
    if kernel_type == 'Matern':
        kernel = Matern(length_scale=0.5, nu=2.5)
    elif kernel_type == 'RBF':
        kernel = RBF(length_scale=0.5)
    else:
        raise ValueError("Invalid kernel type. Choose 'Matern' or 'RBF'")

    # 가우시안 프로세스 모델 설정
    gpr = GaussianProcessRegressor(kernel=kernel, alpha=1e-6)

    for i in range(n_iterations):
        # 모델 학습
        gpr.fit(X_sample, Y_sample)

        # 획득 함수 계산 (Expected Improvement 기반)
        ei = expected_improvement(X, X_sample, Y_sample, gpr)
        X_next = X[np.argmax(ei)]

        # 새로운 데이터 추가(Acquisition Function)
        Y_next = objective_function(X_next.reshape(1, -1))
        X_sample = np.vstack((X_sample, X_next.reshape(-1, 1)))
        Y_sample = np.vstack((Y_sample, Y_next.reshape(-1, 1)))

        # 시각화 (최적화 진행 과정 확인)
        plt.figure(figsize=(8, 5))
        mu, sigma = gpr.predict(X, return_std=True)
        plt.plot(X, objective_function(X), 'r--', label='True Function')
        plt.plot(X, mu, 'b-', label='GP Mean')
        plt.fill_between(X.ravel(), mu - 1.96 * sigma, mu + 1.96 * sigma, alpha=0.2, color='blue')
        plt.scatter(X_sample, Y_sample, c='black', label='Samples')
        plt.axvline(X_next, color='green', linestyle='--', label='Next Sample')
        plt.title(f'Iteration {i+1} (Kernel: {kernel_type})')
        plt.legend()
        plt.show()

# 실행 예시 (Matern과 RBF 커널 비교)
bayesian_optimization(kernel_type='Matern')  # Matern 커널 사용
bayesian_optimization(kernel_type='RBF')  # RBF 커널 사용

- xgboost 하이퍼 파라미터 최적화

In [ ]:
#import packages
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris

#Loading iris dataset from sklearn
iris = load_iris()

#independent feautres
X = iris.data

# target features
y = iris.target

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2, stratify=y)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import xgboost as xgb

# MAPE Metric
def mean_absolute_percentage_error(y_test, y_pred):
    y_test, y_pred = np.array(y_test), np.array(y_pred)
    return np.mean(np.abs((y_test - y_pred) / y_test)) * 100

# 탐색 대상 함수 (XGBRegressor)
def XGB_cv(max_depth,learning_rate, n_estimators, gamma
            ,min_child_weight, subsample
            ,colsample_bytree, silent=True, nthread=-1):

    # 모델 정의
    model = xgb.XGBRegressor(max_depth=int(max_depth),
                            learning_rate=learning_rate,
                            n_estimators=int(n_estimators),
                            gamma=gamma,
                            min_child_weight=min_child_weight,
                            subsample=subsample,
                            colsample_bytree=colsample_bytree,
                            nthread=nthread
                            )
    # 모델 훈련
    model.fit(X_train, y_train)

    # 예측값 출력
    y_pred= model.predict(X_test)

    # 각종 metric 계산
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    # 오차 최적화로 사용할 metric 반환
    return r2

In [ ]:
%pip install bayesian-optimization

In [ ]:
#  bayesian-optimization 라이브러리의 BayesianOptimization 클래스 import
from bayes_opt import BayesianOptimization
import numpy as np

# 실험해보고자하는 hyperparameter 집합
pbounds = {'max_depth': (3, 7),
            'learning_rate': (0.01, 0.2),
            'n_estimators': (5000, 10000),
            'gamma': (0, 100),
            'min_child_weight': (0, 3),
            'subsample': (0.5, 1),
            'colsample_bytree' :(0.2, 1)
            }

# Bayesian optimization 객체 생성
# f : 탐색 대상 함수, pbounds : hyperparameter 집합
# verbose = 2 항상 출력, verbose = 1 최댓값일 때 출력, verbose = 0 출력 안함
# random_state : Bayesian Optimization 상의 랜덤성이 존재하는 부분을 통제
bo=BayesianOptimization(f=XGB_cv, pbounds=pbounds, verbose=2, random_state=1 )
##bo.maximize(init_points=2,n_iter=10, acq='ei',xi=0.01)

# 메소드를 이용해 최대화 과정 수행
# init_points :  초기 Random Search 갯수
# n_iter : 반복 횟수 (몇개의 입력값-함숫값 점들을 확인할지! 많을 수록 정확한 값을 얻을 수 있다.)
# acq : Acquisition Function들 중 Expected Improvement(EI) 를 사용 -> 사라짐(2024.04.12)
# xi : exploration 강도 (기본값은 0.0) -> 사라짐(2024.04.12)
bo.maximize(init_points=2, n_iter=10)

# ‘iter’는 반복 회차, ‘target’은 목적 함수의 값, 나머지는 입력값을 나타냅니다.
# 현재 회차 이전까지 조사된 함숫값들과 비교하여, 현재 회차에 최댓값이 얻어진 경우,
# bayesian-optimization 라이브러리는 이를 자동으로 다른 색 글자로 표시하는 것을 확인할 수 있습니다

# 찾은 파라미터 값 확인
print(bo.max)

# 4.Hyperopt

- HyperOpt는 자동화된 하이퍼파라미터 튜닝 프레임워크로서, fmin()이라는 함수 안에는 3가지의 파라미터가 있다:

    - Objective Function: 최소화할 손실 함수
    - Domain Space: 탐색 범위. 베이지안 최적화에서는 이 범위가 각  하이퍼파라미터에 대해 통계 분포를 만들어낸다.
    - Optimization Algorithm : 최적의 조합을 찾기 위한 알고리즘

참고 : https://velog.io/@emseoyk/%ED%95%98%EC%9D%B4%ED%8D%BC%ED%8C%8C%EB%9D%BC%EB%AF%B8%ED%84%B0-%ED%8A%9C%EB%8B%9D

In [ ]:
%pip install scikit-optimize

In [ ]:
#importing packages

from hyperopt import hp,fmin, tpe, Trials

from hyperopt.pyll.base import scope

from functools import partial

from skopt import space

from skopt import gp_minimize

#defining a method that will perfrom a 5 split cross validation over

#dataset and and will produce the optimum value of the accuracy

def optimize(params, x,y):

    clf = XGBClassifier(**params)

    kf = model_selection.StratifiedKFold(n_splits=5)

    accuracies = []

    for idx in kf.split(X=x,y=y):

        train_idx,test_idx = idx[0],idx[1]

        xtrain = x[train_idx]

        ytrain = y[train_idx]

        xtest = x[test_idx]

        ytest = y[test_idx]

        clf.fit(xtrain,ytrain)

        preds =  clf.predict(xtest)

        fold_acc = metrics.accuracy_score(ytest,preds)

        accuracies.append(fold_acc)

    return -1.0 * np.mean(accuracies)

#defining a set of values as hp for hyperparameters

param_space = {

    "max_depth" : scope.int(hp.quniform("max_depth",3,20, 1)) ,

    "min_child_weight" : scope.int(hp.quniform("min_child_weight",1,8, 1)),

    "n_estimators": scope.int(hp.quniform("n_estimators",100,1500,1)),

    'learning_rate': hp.uniform("learning_rate",0.01,1),

    'reg_lambda': hp.uniform("reg_lambda",0.01,1),

    'gamma': hp.uniform("gamma",0.01,1),

    'subsample': hp.uniform("subsample",0.01,1)

    }

#defiing optimization_fuction as partial and calling optimize within it

optimization_fuction = partial(optimize,x = X, y = y)

trials = Trials()

#Getting the optimum values for hyperparameters

result = fmin(

    fn = optimization_fuction,

    space = param_space,

    algo = tpe.suggest,

    max_evals = 15,

    trials = trials

)

#Printing the best hyperparemeter set

print(result)

# 5.Optuna

- Optuna는 ML 알고리즘의 하이퍼파라미터 튜닝을 자동화해주는 오픈소스 툴입니다. 유사한 툴로 Hyperopt가 있지만 사용성과 문서화, 시각화 제공 여부 등에서 Optuna의 손을 들어주는 경우가 많음.

- 하이퍼파라미터 튜닝에 쓰고 있는 최신 Automl 기법입니다.
- 빠르게 튜닝이 가능하다는 장점이 있음.
- 하이퍼파라미터 튜닝 방식을 지정할수 있다. -> 직관적인 api인 튜닝된 lightgbm도 제공해줍니다.

- 다른 라이브러리들에 비해 직관적인 장점이 있어 코딩하기 용이합니다.


- 거의 모든 ML/DL 프레임워크에서 사용 가능한 넓은 범용성을 가지고 있다.
간단하고 빠르다.
- 최신 동향의 다양한 최적화 알고리즘을 갖추고 있다.
- 병렬 처리가 가능하다.
- 간단한 메소드로 시각화가 가능하다.

- Optuna를 이해하기 위해서는 다음의 용어에 익숙해져야 한다.

    - Study: 목적 함수에 기반한 최적화
    - Trial: 목적함수 시행
- 쉽게 말해 study는 최적화를 하는 과정이고, trial은 다양한 조합으로 목적함수를 시행하는 횟수를 뜻한다.
- Study의 목적은 여러 번의 trial을 거쳐 최적의 하이퍼파라미터 조합을 찾는 것이라고 할 수 있겠다.

Optuna는 **베이지안 최적화(Bayesian Optimization)** 기반의 하이퍼파라미터 자동 탐색 프레임워크

핵심은 “이전 실험 데이터를 바탕으로, 다음 탐색할 파라미터를 확률적으로 예측”하는 것.

---

**Optuna의 구조**

| 구성요소 | 역할 |
|----------|------|
| **Trial** | 하나의 실험 단위. 특정 파라미터 조합에서 모델을 학습·평가 |
| **Sampler** | 다음 시도할 파라미터를 제안 (베이지안 원리 기반) |
| **Pruner** | 비효율적인 Trial 조기 종료 (early stopping) |
| **Storage** | 실험 기록 저장 (기본 SQLite, 또는 DB) |

---

**Bayesian Optimization의 기본 아이디어**

- **목표:**  
  어떤 함수 $f(x) $의 최대(또는 최소)값을 찾기 위해 가능한 적은 평가 횟수로 최적점을 탐색.

- **핵심 원리:**  
  과거의 실험 결과를 이용해 $p(y \mid x) $를 추정하고, 새로운 후보 $ x $를 Expected Improvement(EI)가 가장 큰 지점에서 선택한다.

---

**TPE (Tree-structured Parzen Estimator)**

Optuna의 기본 샘플러는 **TPE**이며, 이는 확률밀도 추정 기반의 방법이다.


#### 1. 데이터 분리

성능 기준 $ y^* $을 기준으로 데이터를 두 그룹으로 나눈다.

$$
l(x) = p(x \mid y < y^*) \quad (\text{좋은 구간})
$$

$$
g(x) = p(x \mid y \ge y^*) \quad (\text{나쁜 구간})
$$

#### 2. 새로운 후보 선택

Expected Improvement(EI)를 근사하기 위해 다음 식을 최대화한다.

$$
\text{maximize} \quad \frac{l(x)}{g(x)}
$$

즉, 좋은 파라미터가 자주 등장했던 영역을 중심으로 탐색하면서 새로운 영역도 일정 비율로 탐험한다(exploration).

---

**Sampler 종류**

| 샘플러 | 특징 | 비고 |
|---------|------|------|
| **TPE** | 기본값. 비선형/비가우시안 목적함수에 강함 | 범용 |
| **RandomSampler** | 무작위 탐색 | baseline 비교용 |
| **CmaEsSampler** | 연속형 파라미터에 유리 | CMA-ES 진화전략 기반 |
| **GridSampler** | 고정된 격자 탐색 | 반복실험용 |
| **BoTorchSampler** | GP 기반 BoTorch 사용 | 고정밀 Bayesian 방식 |


- CMA-ES는 경사를 모르거나 계산하기 어려운 복잡한 함수의 최소값을 찾기 위한,
통계적 자기학습 기반 진화 최적화 알고리즘
---

**Pruner (조기 중단)**

Optuna는 학습 중간에 성능이 낮은 실험을 자동으로 종료시킨다.

| Pruner | 동작 원리 |
|---------|------------|
| **MedianPruner** | 중간 성능이 과거 중앙값보다 낮으면 중단 |
| **SuccessiveHalvingPruner** | 일정 비율로 성능 낮은 trial 제거 |
| **HyperbandPruner** | 자원을 효율적으로 분배하여 탐색 가속 |

---

**수학적 관점**

TPE는 후방분포를 다음처럼 근사한다.

$$
p(y \mid x) \propto p(x \mid y) \, p(y)
$$

이를 반대로 표현하여 효율적으로 샘플링한다.

$$
x^* = \arg\max_x \frac{l(x)}{g(x)}
$$

즉, **EI를 직접 계산하지 않고 밀도비로 EI를 근사**한다.


- 직관적인 코드(단변수 함수)

In [ ]:
%pip install optuna

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import optuna

# 목적 함수 (최적화할 함수)
# 이 함수는 우리가 최소화하려는 함수로, x에 대해 비선형적인 패턴을 가짐
def objective_function(x):
    return np.sin(3*x) + x**2 - 0.7*x

# Optuna를 활용한 최적화 예시
def optuna_optimization():
    # Optuna의 최적화 대상이 될 목적 함수 정의
    def objective(trial):
        # x 값을 -1에서 2 사이의 연속적인 실수 값으로 탐색하도록 설정
        x = trial.suggest_float("x", -1, 2)
        return objective_function(x)

    # Optuna의 Study 객체 생성 (최소화 문제로 설정)
    study = optuna.create_study(direction="minimize")

    # 최적화를 수행하며, 총 20번의 탐색을 실행
    study.optimize(objective, n_trials=20)

    # 최적해 출력 (최적의 x 값과 해당하는 목적 함수 값)
    print("Best value:", study.best_value)  # 최소화된 함수 값 출력
    print("Best parameter:", study.best_params)  # 최적의 x 값 출력

    # 최적화 진행 과정 시각화 (목적 함수 값이 어떻게 줄어드는지 확인)
    fig = optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.title("Optimization History")
    plt.show()

    # 탐색된 파라미터의 중요도 분석 (어떤 변수가 최적화에 가장 중요한 영향을 미치는지 확인)
    fig = optuna.visualization.matplotlib.plot_param_importances(study)
    plt.title("Parameter Importance")
    plt.show()

# Optuna 최적화 실행 (함수 호출 시 최적화를 시작)
optuna_optimization()

- 직관적인 이해(다변수)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import optuna

# 목적 함수 (최적화할 함수)
# 이 함수는 우리가 최소화하려는 함수로, x에 대해 비선형적인 패턴을 가짐
def objective_function(x, y):
    return np.sin(3*x) + y**2 - 0.7*x*y

# Optuna를 활용한 다변수 최적화 예시
def optuna_optimization():
    # Optuna의 최적화 대상이 될 목적 함수 정의
    def objective(trial):
        # x 값을 -1에서 2 사이의 연속적인 실수 값으로 탐색하도록 설정
        x = trial.suggest_float("x", -1, 2)
        y = trial.suggest_float("y", -2, 2)
        return objective_function(x, y)

    # Optuna의 Study 객체 생성 (최소화 문제로 설정)
    study = optuna.create_study(direction="minimize")

    # 최적화를 수행하며, 총 50번의 탐색을 실행 (다변수 탐색 강화)
    study.optimize(objective, n_trials=50)

    # 최적해 출력 (최적의 x, y 값과 해당하는 목적 함수 값)
    print("Best value:", study.best_value)  # 최소화된 함수 값 출력
    print("Best parameters:", study.best_params)  # 최적의 x, y 값 출력

    # 최적화 진행 과정 시각화 (목적 함수 값이 어떻게 줄어드는지 확인)
    fig = optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.title("Optimization History")
    plt.show()

    # 탐색된 파라미터의 중요도 분석 (어떤 변수가 최적화에 가장 중요한 영향을 미치는지 확인)
    fig = optuna.visualization.matplotlib.plot_param_importances(study)
    plt.title("Parameter Importance")
    plt.show()

    # x와 y 값의 탐색 과정 시각화 (탐색 경로 분석)
    fig = optuna.visualization.matplotlib.plot_contour(study, params=["x", "y"])
    plt.title("Parameter Contour")
    plt.show()

# Optuna 최적화 실행 (다변수 최적화를 수행)
optuna_optimization()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import optuna

# 목적 함수 (최적화할 함수)
# 이 함수는 우리가 최소화하려는 함수로, x에 대해 비선형적인 패턴을 가짐
def objective_function(x, y):
    return np.sin(3*x) + y**2 - 0.7*x*y

# Optuna를 활용한 다변수 최적화 예시 (고급 기능 추가)
def optuna_optimization():
    # Optuna의 최적화 대상이 될 목적 함수 정의
    def objective(trial):
        # x 값을 -1에서 2 사이의 연속적인 실수 값으로 탐색하도록 설정
        x = trial.suggest_float("x", -1, 2)
        y = trial.suggest_float("y", -2, 2)
        return objective_function(x, y)

    # Optuna의 Study 객체 생성 (최소화 문제로 설정)
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler())
    #TPE (Tree-structured Parzen Estimator) 기반 샘플링 방법, 기존 샘플 데이터로부터 확률 모델을 학습하여, 더 좋은 후보값을 예측
    #기존의 무작위 샘플링보다 빠르고 효율적인 탐색이 가능 / 특히, 하이퍼파라미터 튜닝에서 많이 사용됨

    # 최적화를 수행하며, 총 100번의 탐색을 실행 (탐색 성능 향상)
    study.optimize(objective, n_trials=100)

    # 최적해 출력 (최적의 x, y 값과 해당하는 목적 함수 값)
    print("Best value:", study.best_value)  # 최소화된 함수 값 출력
    print("Best parameters:", study.best_params)  # 최적의 x, y 값 출력

    # 최적화 진행 과정 시각화 (목적 함수 값이 어떻게 줄어드는지 확인)
    fig = optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.title("Optimization History")
    plt.show()

    # 탐색된 파라미터의 중요도 분석 (어떤 변수가 최적화에 가장 중요한 영향을 미치는지 확인)
    fig = optuna.visualization.matplotlib.plot_param_importances(study)
    plt.title("Parameter Importance")
    plt.show()

    # x와 y 값의 탐색 과정 시각화 (탐색 경로 분석)
    fig = optuna.visualization.matplotlib.plot_contour(study, params=["x", "y"])
    plt.title("Parameter Contour")
    plt.show()

    # 하이퍼파라미터의 조합을 3D 산점도로 시각화
    trials = np.array([[t.params["x"], t.params["y"], t.value] for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(trials[:, 0], trials[:, 1], trials[:, 2], c=trials[:, 2], cmap='viridis', marker='o')
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("Objective Value")
    plt.title("Optimization Search Space")
    plt.show()

# Optuna 최적화 실행 (다변수 최적화를 수행)
optuna_optimization()

- randomsampler VS TPEsampler

In [ ]:
import optuna
import numpy as np

# 최적화할 목적 함수 정의
def objective(trial):
    x = trial.suggest_float("x", -5, 5)
    y = trial.suggest_float("y", -5, 5)
    return (x - 2) ** 2 + (y + 3) ** 2  # (2, -3)에서 최솟값을 가짐

# Random Sampler (무작위 탐색)
study_random = optuna.create_study(direction="minimize", sampler=optuna.samplers.RandomSampler())
study_random.optimize(objective, n_trials=50)
print("Random Sampler Best Value:", study_random.best_value, "Best Params:", study_random.best_params)

# TPE Sampler (TPE 기반 탐색)
study_tpe = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler())
study_tpe.optimize(objective, n_trials=50)
print("TPE Sampler Best Value:", study_tpe.best_value, "Best Params:", study_tpe.best_params)


-  DashBorad 연결

In [ ]:
%pip install optuna

In [ ]:
import optuna
import numpy as np
import xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 불필요한 로그 출력 억제 (Optuna는 trial별 경고가 많음)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# scikit-learn 내장 데이터 (유방암 이진 분류)
X, y = load_breast_cancer(return_X_y=True)

# StratifiedKFold: 클래스 비율 유지하면서 데이터를 5등분 교차검증
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# GPU 감지 함수
def get_device():
    try:
        # CUDA로 테스트 학습 시도
        model = xgb.XGBClassifier(tree_method="hist", device="cuda", n_estimators=1)
        model.fit([[0, 0], [1, 1]], [0, 1])  # 매우 작은 데이터로 시험
        return "cuda"
    except XGBoostError:
        return "cpu"

device = get_device()
print("감지된 디바이스:", device)

In [ ]:
def objective(trial):
    """
    Optuna가 각 trial(실험)마다 호출하는 목적 함수.
    trial 객체가 하이퍼파라미터 후보를 제안하고,
    그 조합으로 학습한 모델의 성능(AUC)을 평가한다.
    """
    # 하이퍼파라미터 탐색 범위 정의
    params = {
        # 트리 깊이
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        # 학습률(learning_rate): 로그스케일로 탐색 (1e-3 ~ 0.3)
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        # 노드 분할 시 최소 자식 가중치 (클수록 보수적인 분할)
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        # 샘플 비율 (bagging)
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        # 피처 비율 (feature sampling)
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        # L2 정규화 파라미터
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        # L1 정규화 파라미터
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        # 트리 개수 (boosting 단계)
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        # 이진 분류 문제 설정
        "objective": "binary:logistic",
        "eval_metric": "auc",  # 평가 지표: AUC (Area Under Curve)
        "tree_method": "hist",
        "device" : device,
        "n_jobs": -1,          # 모든 CPU 코어 사용
        "random_state": 42,    # 재현성 보장
    }

    # 지정된 파라미터로 XGBoost 모델 생성
    model = xgb.XGBClassifier(**params)

    # 5겹 교차검증으로 평균 AUC 계산
    auc = cross_val_score(model, X, y, cv=cv, scoring="roc_auc").mean()

    # Optuna는 "값을 최소화"하도록 설계되어 있으므로, -AUC 반환
    # (즉, AUC를 최대화하는 것이 목표)
    return -auc


In [ ]:
# 최적화 실행

# Optuna study 생성
# - direction="minimize": 반환값(-AUC)을 최소화
# - study_name: 대시보드에서 식별용 이름
study = optuna.create_study(direction="minimize", study_name="xgb-dashboard")

# 50번의 trial 실행 (즉, 50가지 하이퍼파라미터 조합 평가)
study.optimize(objective, n_trials=50)

# 결과 출력
print("Best AUC:", -study.best_value)  # 음수 부호 복원
print("Best params:", study.best_params)


In [ ]:
from optuna.visualization.matplotlib import (
    plot_parallel_coordinate, plot_contour, plot_slice, plot_param_importances
)
import matplotlib.pyplot as plt

plot_param_importances(study); plt.show()
plot_parallel_coordinate(study); plt.show()
plot_contour(study, params=["max_depth","learning_rate"]); plt.show()
plot_slice(study); plt.show()

- Optuna는 실험 로그를 시각적으로 확인할 수 있는 대시보드(dashboard) 기능을 제공

- Colab에서는 optuna-dashboard 패키지와 ngrok을 함께 이용해 웹 인터페이스를 띄울 수 있습니다.
    - ngrok은 로컬 개발 환경에서 인터넷을 통해 웹 애플리케이션에 안전하게 접근할 수 있도록 해주는 도구
- Optuna 대시보드는 SQLite DB 파일을 기반으로 작동합니다. 현재 Colab 세션의 study 객체를 .db 파일로 저장합니다.

In [ ]:
%pip install optuna-dashboard pyngrok

In [ ]:
import optuna

# study를 SQLite 파일로 저장
storage_name = "sqlite:///xgb_dashboard.db"
study = optuna.create_study(
    direction="minimize",
    study_name="xgb-dashboard",
    storage=storage_name
)

# 다시 동일 objective로 학습 (간단히 10회만)
study.optimize(objective, n_trials=10)

print("Dashboard용 study 저장 완료:", storage_name)

- Colab은 로컬 서버에 직접 접근할 수 없기 때문에, ngrok을 사용하여 외부 접속 가능한 URL을 생성합니다.

**cloudflared (간단 사용 가이드)**


cloudflared는 로컬 서버를 공용 URL로 노출할 때 쓰는 경량 터널 도구입니다 ngrok과 달리 토큰 불필요.

In [ ]:
# 최신 linux-amd64 바이너리 설치
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# 간단한 Optuna Dashboard + cloudflared 터널 실행 (Colab에서 한 셀로 실행)
import threading, time, subprocess, re, sys
import optuna_dashboard

def start_dashboard_and_tunnel(port=8082, dburl="sqlite:///xgb_dashboard.db",
                               cloudflared_path="/usr/local/bin/cloudflared"):
    # 1) 대시보드(비차단) 시작
    threading.Thread(target=lambda: optuna_dashboard.run_server(dburl, host="0.0.0.0", port=port),
                     daemon=True).start()
    time.sleep(1)  # 서버가 뜰 시간을 약간 둠
    print(f"Local: http://0.0.0.0:{port}")

    # 2) cloudflared 프로세스 실행 및 로그에서 public URL 추출
    proc = subprocess.Popen([cloudflared_path, "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"],
                             stdout=subprocess.%pipE, stderr=subprocess.STDOUT, text=True)
    public_url = None
    for _ in range(300):                      # 최대 대기 루프(약 몇 초)
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.05); continue
        sys.stdout.write(line)                # cloudflared 로그를 실시간으로 출력
        m = re.search(r"(https://[-a-zA-Z0-9.]+\.trycloudflare\.com)", line)
        if m:
            public_url = m.group(1)
            break

    print("\n외부 접속 URL:", public_url or "로그에서 trycloudflare 주소 확인 필요")
    return proc, public_url

# 호출 (한 번만 실행)
proc, public_url = start_dashboard_and_tunnel(port=8082, dburl="sqlite:///xgb_dashboard.db")

- 업그레이드 버전

- Optuna Integration (optuna-integration)
    - Optuna와 다양한 머신러닝 프레임워크(XGBoost, LightGBM, PyTorch 등)를 쉽게 연동하는 라이브러리
    - optuna.integration.lightgbm.LightGBMPruningCallback을 통해 LightGBM과 Optuna를 쉽게 연결 가능.

In [ ]:
%pip install optuna-integration

- “Optuna를 활용한 LightGBM 분류기의 자동 하이퍼파라미터 탐색과 조기 중단(Pruning)”

In [ ]:
import optuna
import lightgbm as lgb
from optuna.integration import LightGBMPruningCallback
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

# 데이터 로드
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

def objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 20, 50),
        "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.1),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=100),
    }

    model = lgb.LGBMClassifier(**params)

    # 조기 종료 및 Pruning 설정
    callbacks = [
        LightGBMPruningCallback(trial, "binary_logloss"),  # log loss가 감소하지 않으면 pruning
        lgb.early_stopping(50),  # 50회 동안 개선이 없으면 조기 종료
    ]

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric="logloss",
        callbacks=callbacks  # early_stopping_rounds 대신 사용
    )

    y_pred_prob = model.predict_proba(X_test)[:, 1]  # 클래스 1에 대한 확률
    return log_loss(y_test, y_pred_prob)  # log loss 값 반환 (작을수록 좋음)

# Optuna 최적화 실행 (`direction="minimize"`로 변경)
study = optuna.create_study(direction="minimize")  # log loss는 낮을수록 좋음
study.optimize(objective, n_trials=50)

# 최적의 하이퍼파라미터 출력
print("Best parameters:", study.best_params)

In [ ]:
best_params = study.best_params

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X_train, y_train,
                eval_set=[(X_test, y_test)],
                eval_metric="logloss",
                callbacks=[lgb.early_stopping(50)])

y_prob = final_model.predict_proba(X_test)[:, 1]
y_hat = (y_prob >= 0.5).astype(int)

from sklearn.metrics import log_loss, accuracy_score, roc_auc_score
print({
    "logloss": round(log_loss(y_test, y_prob), 5),
    "accuracy": round(accuracy_score(y_test, y_hat), 5),
    "auc": round(roc_auc_score(y_test, y_prob), 5)
})

**Optuna는 define-by-run 스타일을 제공해서, Python 코드 안에서 조건문으로 탐색공간을 동적으로 만들 수 있음**

Optuna의 핵심은 크게 두 축.

1. 어떤 하이퍼파라미터를 시도할지 고르는 것
2. 가망이 낮은 실험을 빨리 중단하는 것

### 1. Sampler란 무엇인가

- Sampler는 각 trial에서 사용할 하이퍼파라미터 값을 어떤 전략으로 제안할지 결정하는 구성요소.


- trial.suggest_float(...), trial.suggest_int(...), trial.suggest_categorical(...)가 호출될 때, 실제로 어떤 값을 줄지 내부적으로 판단하는 주체가 sampler.

- Optuna의 sampler 모듈은 여러 sampling strategy를 구현한 클래스들로 구성

- 좋은 sampler는 이 정보를 이용해 유망한 구간을 더 자주 탐색하고, 비효율적인 구간은 덜 보게 만듦.

    - objective 함수: “이 조합이 얼마나 좋은가?”를 평가
    - sampler: “다음에는 어떤 조합을 시도할까?”를 결정

**sampler의 역할은 단순 랜덤 시도가 아니라,
이전 실험 결과를 이용해 더 똑똑하게 다음 실험을 고르는 것**

In [ ]:
# sampler는 단순히 고정된 표를 탐색하는 것이 아니라 현재 코드 흐름에서 실제로 열린 탐색공간
import torch
import torch.nn as nn
import torch.optim as optim
import optuna

# 이 함수 내부가 바로 '실제로 열린 탐색 공간'이 됩니다.
def objective(trial):
    # (탐색 공간 정의) 코드 흐름 중간에 실시간으로 제안받음
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)      # 학습률 탐색 (로그 스케일)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"]) # 최적화 도구 선택
    hidden_dim = trial.suggest_int("hidden_dim", 32, 256, step=32) # 은닉층 크기 선택

    # 제안받은 파라미터로 모델 및 최적화 도구 생성
    model = nn.Sequential(
        nn.Linear(10, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, 2)
    )

    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # 간이 학습 루프 (코드 흐름 내에서 평가).
    dummy_input = torch.randn(16, 10)
    dummy_target = torch.randint(0, 2, (16,))

    model.train()
    for epoch in range(5): # 실습을 위해 짧게 실행
        optimizer.zero_grad()
        output = model(dummy_input)
        loss = criterion(output, dummy_target)
        loss.backward()
        optimizer.step()

        # 중간에 가망이 없으면 학습을 중단하는 Pruning(가지치기)
        trial.report(loss.item(), epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    # 검증 결과(Loss 등)를 반환하여 Optuna가 다음 시도를 결정하게 함
    return loss.item()

# 최적화 실행 (Study 생성)
study = optuna.create_study(direction="minimize") # Loss를 최소화하는 방향
study.optimize(objective, n_trials=20) # 총 20번의 서로 다른 조합 시도

print(f"최적의 파라미터: {study.best_params}")
print(f"최저 Loss: {study.best_value}")

- getattr: 동적으로 접근할 떄 사용하는 함수
    - 코드를 작성할 때 속성 이름을 알 수 없거나, 문자열로 속성을 가져와야 하는 상황에서 필수적

In [ ]:
class Character:
    def __init__(self, name,level):
        self.name = name
        self.level = level
hero = Character("파이썬 전사", 10)

# 일반적인 접근
print(hero.name) # 출력: 파이썬 전사


# getattr
name_attr = "name"
print(getattr(hero, name_attr)) # 출력: 파이썬 전사

# 없는 속성에 접근할때,
hp = getattr(hero,'hp',100)
print(f"HP: {hp}")


In [ ]:
 # getattr는 변수뿐만 아니라 함수(메소드)도 가져올 수 있음.

class Tools:
    def pen(self):
        return "펜으로 그림을 그립니다."

    def eraser(self):
        return "지우개를 지웁니다"
tool = Tools()
action = input("사용할 도구를 입력하세요 (pen/earser):")

func = getattr(tool, action,None)

if func:
    print(func())
else:
    print("해당하는 도구가 없습니다.")

 위와 비슷한 함수는 hasattr,getattr,setattr,delattr etc

 - 동기(Synchronous)방식
    - 빨래가 다 될때까지 세탁기 앞에서 아무것도 안하고 서서 기다리는 것과 비슷

In [ ]:
import time

def deliver_pizza(n):
    print(f"피자 {n}번 배달 시작...")
    time.sleep(2) # 배달 중 (2초 대기)
    print(f"피자 {n}번 배달 완료!")

def main_sync():
    start_time = time.time()

    # 3번의 배달을 순차적으로 진행
    deliver_pizza(1)
    deliver_pizza(2)
    deliver_pizza(3)

    end_time = time.time()
    print(f"--- 동기 방식 총 소요 시간: {end_time - start_time : .2f}초 ---")
main_sync()

- 비동기(ASynchronous)
    - 세탁기를 돌려놓고 거실에서 책을 읽는 것과 같은
    - 기다리는 동안 다른 일을 하거나, 다른 작업을 동시에 시작
    - 여러 작업을 동시에(Non-blocking) 처리


**기다리는 시간이 많은 작업**에서 유리

- 비동기가 큰 의미가 없는 경우
    - 단순 계산만 하는 코드
    - CPU 연산이 아주 많은 작업
    - 작은 스크립트

- 행렬 계산, 딥러닝 학습 CPU/GPU 계산이 중심인 경우네는 비동기보다는 멀티프로세싱, 벡터화, GPU 활용

In [ ]:
import asyncio
import time

async def deliver_pizza_async(n):
    print(f"비동기 피자 {n}번 배달 시작...")
    await asyncio.sleep(2) # 배달 중 (2초 대기)
    print(f"피자 {n}번 배달 완료!")

async def main_async():
    start_time = time.time()

    # 3개의 작업을 동시에 예약
    await asyncio.gather(
        deliver_pizza_async(1),
        deliver_pizza_async(2),
        deliver_pizza_async(3)
    )

    end_time = time.time()
    print(f"--- 비동기 방식 총 소요 시간: {end_time - start_time : .2f}초 ---")
# 비동기 루프 실행
await main_async()
#asyncio.run(main_async())

In [ ]:
# 동적 함수 실행
## getattr로 가져온 함수가 무엇이든 실행해주는 코드
import asyncio # 비동기 처리를 위한 라이브러리
import inspect # 객체의 정보를 조사(동기/비동기 판별)등하기 위한 라이브러리

class ActionHandler:
    #일반적인 동기 메서드
    def say_hello(self,name):
        return f"안녕하세요, {name}님!"
    #async 키워드가 붙은 비동기 메서드(코루틴)
    async def fetch_dat(self, db_name):
        await asyncio.sleep(1) # 비동기방식으로 1초기 대기(I/O 작업 시뮬레이션)
        return f"[{db_name}]"

async def smart_executor(handler, func_name, *args):
    # getattr를 사용해 hadnler 객체에서 func_name이라는 속성을 가져옴
    ## 해당 이름이 없으면 None 반환하도록 설정
    func = getattr(handler, func_name, None)

    # 함수가 존재하지 않을 경우의 예외 처리
    if not func:
        print(f"오류: '{func_name}' 함수를 찾을 수 없습니다.")
        return

    # inspect.iscoroutinefunction 사용하여 가져온 함수가 'async def'인지 확인
    if inspect.iscoroutinefunction(func):
        #비동기 함수라면 반드시 await를 붙여서 실행
        print(f"(시스템: {func_name}은 비동기 함수 입니다. await를 사용합니다.)")
        result = await func(*args)
    else:
        #일반 함수라면 바로 호출하여 실행
        print(f"(시스템: {func_name}은 동기 함수입니다. 즉시 실행합니다.)")
        result = func(*args)
    #최종 실행 결과 출력
    print(f"결과: {result}")

handler = ActionHandler()

async def main():
    print("---첫 번째 호출: 동기 함수 테스트:---")
    # 문자열 "say hello"를 전달하여 동적으로 함수를 찾아 실행함.
    await smart_executor(handler, "say_hello", "Gemini")

    print("---두 번째 호출: 비동기 함수 테스트:---")
    await smart_executor(handler, "fetch_data", "USER_DB")

#비동기 루프 시작
if __name__ =="__main__":
    try:
        asyncio.run(main())
    except RuntimeError:
        # 이미 루프가 실행 중인 환경(jupyter 등)을 위한 예외처리
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.run(main())



 - FastAPI, Django(웹 프로그래밍) 등에서 쓰려고 함.
    - 사용자가 /login들어오면 getattr(contorller, "login")으로 함수를 찾고, 그 함수 async def인지 확인
    - API 게이트웨이: 외부 요청에 따라 동기적으로 비동기적일지 결정을 해야함.

### 대표 Sampler

### TPESampler(Tree-structured Parzen Estimator)

https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.TPESampler.html

TPE는 직관적으로 보면:

- 성능이 좋았던 trial들의 분포
- 성능이 나빴던 trial들의 분포

를 비교해서, 좋은 성능이 나올 가능성이 높은 구간을 더 선호하도록 다음 값을 제안. -> 일반적인 하이퍼파라미터 튜닝에서 매우 자주 사용

- 연속형, 정수형, 범주형 파라미터를 두루 다루기 좋음
- 조건부 탐색공간과도 잘 맞음
- 기본 선택으로 쓰기 좋음
- tabular ML, boosting 계열에서 자주 잘 먹힘


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna

# 학습할 함수(목적 함수) 정의
def objective(trial):
    # [탐색 공간] TPESampler가 이 구간 안에서 영리하게 값을 제안.
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    hidden_size = trial.suggest_int("hidden_size", 16, 128)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop"])

    # 간단한 모델 구성
    model = nn.Sequential(
        nn.Linear(10, hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size, 1)
    )

    # 제안받은 이름으로 옵티마이저 생성
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    # 간이 학습 과정 (100번의 데이터 업데이트)
    for _ in range(100):
        inputs = torch.randn(32, 10)
        target = torch.randn(32, 1)

        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

    # 최종 손실값을 반환 (Optuna는 이 값을 최소화하려고 노력함)
    return loss.item()

# TPESampler 설정 및 최적화 실행
# sampler 인자에 TPESampler를 넣어줍니다. (생략해도 기본값으로 적용됨)
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler()
)

# 3. 20번의 서로 다른 조합 시도
study.optimize(objective, n_trials=20)

# 4. 결과 출력
print(f"최적의 파라미터: {study.best_params}")

### RandomSampler

- RandomSampler는 말 그대로 무작위로 하이퍼파라미터를 제안.
- 가장 단순하지만, 기준선으로는 매우 중요.

- 복잡한 sampler를 썼더니 정말 좋아진 것인지 확인하려면 랜덤 탐색과 비교해보는 것이 가장 기본적인 점검

     - 구현과 해석이 가장 단순
     - 초기 디버깅에 좋음
     - baseline으로 자주 사용

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna

# 목적 함수 정의 (평가할 모델과 학습 루프)
def objective(trial):
    # [탐색 공간] RandomSampler는 이 범위 내에서 무작위로 값을 추출합니다.
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    dropout_rate = trial.suggest_float("dropout", 0.1, 0.5)

    # 간단한 분류 모델
    model = nn.Sequential(
        nn.Linear(20, 64),
        nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(64, 2)
    )

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # 가상 데이터로 1단계 학습 진행
    inputs = torch.randn(batch_size, 20)
    target = torch.randint(0, 2, (batch_size,))

    optimizer.zero_grad()
    output = model(inputs)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()

    # Loss 값을 반환 (이 값이 낮을수록 좋은 파라미터 조합)
    return loss.item()

# RandomSampler 설정 및 최적화 실행
# seed를 고정하면 매번 같은 무작위 순서로 탐색합니다.
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.RandomSampler(seed=42)
)

# 15번의 무작위 시도 진행
study.optimize(objective, n_trials=15)

# 3. 결과 확인
print(f"무작위 탐색 중 가장 좋았던 파라미터: {study.best_params}")

### CmaEsSampler는 CMA-ES(Covariance Matrix Adaptation Evolution Strategy)

- 연속형 탐색공간에서 파라미터 간 상관관계를 반영하면서 탐색하는 데 강점
- 안개 속에서 가장 낮은 골짜기를 찾기 위해, 여러 명의 탐험대가 무리를 지어 이동하며 지형을 파악하는 방식
- 다만 범주형 변수가 많거나 조건부 분기가 많은 문제에서는 TPE가 더 다루기 편한 경우가 많음.

#### CMA-ES의 핵심 원리: "진화와 적응"

1. 샘플링 (Generation): 현재 추측하는 중심점 근처에서 여러 개의 파라미터 후보(개체)를 무작위로 뽑습니다.

2. 평가 (Evaluation): 뽑힌 후보들의 성능(Loss 등)을 계산합니다.

3. 업데이트 (Adaptation): 성적이 좋은 후보들이 모여 있는 방향으로 중심점을 이동시키고, 그 주변을 더 넓게 혹은 좁게 탐색하도록 공분산 행렬(탐색 범위의 모양과 크기)을 업데이트.


In [ ]:
# CmaEsSampler: cmaes 패키지 필요
%pip install cmaes

In [ ]:

import optuna
import torch.nn as nn
import torch.optim as optim

def objective(trial):
    # CMA-ES는 이런 연속적인 수치형(float) 탐색에 매우 강합니다.
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    momentum = trial.suggest_float("momentum", 0.0, 1.0)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    # 간단한 모델 및 학습 생략 (반환값은 Loss)
    return 0.5 # 예시 결과값

# CmaEsSampler 설정
# 범주형 파라미터가 포함되어 있다면 warn_independent_sampling=True 설정을 권장합니다.
sampler = optuna.samplers.CmaEsSampler()

study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=30)

print(f"Best Params: {study.best_params}")

### NSGA-II (Non-dominated Sorting Genetic Algorithm II) / multi-objective sampler

NSGA-II는 유전 알고리즘(Genetic Algorithm)을 기반으로 여러 개의 목표를 동시에 최적화.

#### 핵심 원리: 파레토 최적 (Pareto Efficiency)

다목적 최적화에서는 하나의 '정답'이 존재하지 않음.

대신 "어느 하나를 개선하기 위해서는 다른 하나를 희생해야 하는 상태"의 점들의 집합을 찾습니다. 이를 파레토 전선(Pareto Front)이라고 부름.

- 비지배 정렬 (Non-dominated Sorting): 다른 모든 목표에서 자신보다 뛰어난 후보가 없는 '우수한' 후보들을 1순위 그룹으로 분류.

- 밀집도 분석 (Crowding Distance): 파레토 전선 위에서 후보들이 한곳에 몰리지 않고 골고루 퍼지도록 관리하여 다양한 선택지를 제공합니다.

In [ ]:
#다목적 최적화 설정을 감지하면 자동으로 선택
# 정확도 vs 모델 경량화
import optuna
import torch.nn as nn

def objective(trial):
    # 탐색 공간: 은닉층 크기와 학습률
    hidden_size = trial.suggest_int("hidden_size", 16, 128)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    # 1. 첫 번째 목표: 정확도 (Maximize)
    # (실제 학습 코드가 들어갈 자리입니다)
    accuracy = 0.95 - (1.0 / hidden_size) # 예시: 은닉층이 클수록 정확도 상승

    # 2. 두 번째 목표: 모델의 가벼움 (Minimize Params)
    params_count = hidden_size * 10 # 예시: 은닉층이 클수록 파라미터 증가

    # 두 개의 목표를 튜플로 반환합니다.
    return accuracy, params_count

# 다목적 최적화 설정 (목표가 2개이므로 directions 리스트 전달)
study = optuna.create_study(
    directions=["maximize", "minimize"], # 정확도는 최대화, 파라미터는 최소화
    sampler=optuna.samplers.NSGAIISampler()
)

study.optimize(objective, n_trials=50)

# 결과 확인: 파레토 전선에 있는 최적의 후보들 출력
print(f"최적의 후보(Pareto Front) 개수: {len(study.best_trials)}")
for t in study.best_trials:
    print(f"Params: {t.params}, Values: {t.values}")

## Pruning이란 무엇인가

- Pruning은 학습 도중 성능이 좋지 않은 trial을 끝까지 학습하지 않고 중간에 중단하는 기능
- Optuna는 이를 “unpromising trials”를 초기에 자동으로 멈추는 방식으로 설명

- 현재 Optuna의 pruner 모듈은 single-objective optimization을 대상으로 안내되고 있음.


    - Sampler: 다음 trial의 하이퍼파라미터를 정함
    - Pruner: 지금 돌고 있는 trial을 계속할지 멈출지 정함

### 왜 pruning이 중요한가

PyTorch 학습은 보통 trial 하나의 비용이 큼.

- 예를 들면:

    - epoch 수가 많고
    - 모델이 크고
    - 데이터가 많고
    - GPU 자원을 사용하면

trial 하나가 몇 분에서 몇 시간까지 걸릴 수 있습니다.

- 이때 pruning을 쓰면:

    - 성능이 안 좋은 trial을 조기에 중단하고
    - 같은 시간 안에 더 많은 trial을 시도할 수 있고
    - 더 빨리 좋은 하이퍼파라미터를 찾을 수 있습니다. Optuna 공식 튜토리얼도 pruning을 효율적 최적화의 핵심 기능으로 설명하고, 반복 학습 중 report()와 should_prune()를 호출해 활성화하라고 안내.

**pruning의 본질은 시간과 계산 자원을 아끼는 자동 early stopping**

#### 1. MedianPruner

가장 많이 시작하는 기본 pruner입니다.

- 아이디어:

    - 같은 step에서 이전 completed trial들의 성능 중앙값과 비교해서
    - 현재 trial이 너무 나쁘면 중단

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
from torch.utils.data import DataLoader, TensorDataset

# 가상 데이터 생성 (입력 10개 -> 출력 1개 회귀 문제)
X = torch.randn(1000, 10)
y = torch.randn(1000, 1)
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# 간단한 신경망 모델 정의
class SimpleNet(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(10, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.layers(x)

# Optuna 목적 함수 (Objective)
def objective(trial):
    # 탐색 공간: 은닉층 크기와 학습률
    hidden_dim = trial.suggest_int("hidden_dim", 16, 128)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    model = SimpleNet(hidden_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    # --- 학습 루프 ---
    for epoch in range(50):  # 최대 50 에포크 학습
        model.train()
        epoch_loss = 0
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(loader)

        # [핵심] 1. 현재 에포크의 성적을 Optuna에 보고
        trial.report(avg_loss, epoch)

        # [핵심] 2. 현재 성적이 이전 성공작들의 '중간값'보다 나쁘면 학습 중단!
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return avg_loss

# Study 생성 및 실행
# n_startup_trials: 처음 5번은 데이터 수집을 위해 무조건 끝까지 학습
# n_warmup_steps: 각 실험의 초기 10 에포크까지는 지켜본 뒤 가지치기 시작
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
study = optuna.create_study(direction="minimize", pruner=pruner)

study.optimize(objective, n_trials=30)

# 5. 결과 확인
print(f"최적 파라미터: {study.best_params}")
print(f"전체 시도 횟수: {len(study.trials)}")
pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"중간에 가지치기(Pruned) 된 횟수: {len(pruned_trials)}")

#### 2. SuccessiveHalvingPruner

- 이 pruner는 Asynchronous Successive Halving Algorithm 기반.
- 여러 시도(Trial)를 동시에 시작한 뒤, 일정 구간마다 성적이 나쁜 절반을 탈락시키고 살아남은 절반에게만 더 많은 자원(에포크)을 할당하는 매우 효율적인 방식
    - 처음엔 많은 후보를 얕게 본다
    - 잘 되는 trial만 살아남는다
    - 살아남은 trial에 더 많은 epoch를 투자한다

- 장점:

    - 공격적으로 잘라낼 수 있음
    - 많은 trial을 빠르게 거를 수 있음

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna

# 간단한 신경망 모델 정의
def create_model(n_layers, hidden_dim):
    layers = []
    in_dim = 20
    for _ in range(n_layers):
        layers.append(nn.Linear(in_dim, hidden_dim))
        layers.append(nn.ReLU())
        in_dim = hidden_dim
    layers.append(nn.Linear(hidden_dim, 1))
    return nn.Sequential(*layers)

# 목적 함수 (Objective)
def objective(trial):
    # 하이퍼파라미터 제안
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_dim = trial.suggest_int("hidden_dim", 16, 128)
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)

    model = create_model(n_layers, hidden_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    # 가상 데이터 생성
    data = torch.randn(64, 20)
    target = torch.randn(64, 1)

    # 학습 루프 (Successive Halving은 중간 보고가 필수입니다)
    for step in range(100):  # 최대 100 스텝(에포크)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        # [중요] 현재 스텝의 성능을 보고
        trial.report(loss.item(), step)

        # [중요] 가지치기 여부 확인 (토너먼트 탈락 여부)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return loss.item()

# SuccessiveHalvingPruner 설정
# min_resource: 최소한으로 보장할 학습 스텝 (예: 1단계)
# reduction_factor: 다음 단계로 넘어갈 때 살아남을 비율 (3이면 상위 1/3만 생존)
pruner = optuna.pruners.SuccessiveHalvingPruner(
    min_resource=1,
    reduction_factor=3
)

study = optuna.create_study(direction="minimize", pruner=pruner)
study.optimize(objective, n_trials=30)

# 결과 확인
print(f"최적 파라미터: {study.best_params}")
print(f"완료된 시도: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}")
print(f"가지치기 된 시도: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}")

#### 3.HyperbandPruner

**핵심 원리: "다양성과 효율의 조화"**


Successive Halving은 처음에 너무 적은 자원(에포크)만 주고 평가하기 때문에, 나중에 잠재력이 터지는 모델(Late Bloomer)을 성급하게 탈락시킬 위험이 있음.

HyperbandPruner는 이 문제를 해결하기 위해 여러 개의 '토너먼트 대진표(Bracket)'를 동시에 운영:

- 공격적 대진표: 아주 많은 후보를 짧게 보고 빠르게 탈락시킴 (효율 중시)

- 보수적 대진표: 적은 후보를 뽑더라도 처음부터 길게 지켜보고 탈락시킴 (안정성 중시)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna

def objective(trial):
    # 하이퍼파라미터 제안
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    momentum = trial.suggest_float("momentum", 0.0, 0.9)

    model = nn.Sequential(nn.Linear(10, 5), nn.ReLU(), nn.Linear(5, 1))
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    criterion = nn.MSELoss()

    # 가상 데이터
    data, target = torch.randn(32, 10), torch.randn(32, 1)

    # 학습 루프
    # Hyperband는 max_resource(여기서는 81 에포크)까지 단계를 나누어 관리합니다.
    for step in range(81):
        optimizer.zero_grad()
        loss = criterion(model(data), target)
        loss.backward()
        optimizer.step()

        # [보고] 현재 성적 보고
        trial.report(loss.item(), step)

        # [가지치기] Hyperband의 로직에 따라 중단 여부 결정
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return loss.item()

# 3. HyperbandPruner 설정
# min_resource: 최소 학습 단위 (보통 1)
# max_resource: 한 실험에 투자할 최대 학습 단위 (전체 에포크 수와 일치 권장)
# reduction_factor: 각 단계에서 살아남을 비율 (기본값 3)
pruner = optuna.pruners.HyperbandPruner(
    min_resource=1,
    max_resource=81,
    reduction_factor=3
)

study = optuna.create_study(direction="minimize", pruner=pruner)
study.optimize(objective, n_trials=50)

print(f"최적 파라미터: {study.best_params}")

#### 4.PatientPruner

**핵심 원리**

이 Pruner는 다른 Pruner를 내부적으로 감싸서(Wrapping) 작동.

- 작동 방식: 기준이 되는 Pruner(예: MedianPruner)가 "이 실험은 잘라야 해!"라고 신호를 보내더라도, 설정한 patience 횟수만큼은 판단을 유보하고 학습을 더 진행.

- 비유: 성적이 떨어진 학생에게 바로 퇴학 처분을 내리는 게 아니라, "다음 시험까지는 기회를 더 줄게"라고 기다려주는 자애로운 선생님과 같습니다.

**왜 PatientPruner가 필요한가요?**

딥러닝 학습 과정에서는 다음과 같은 상황이 자주 발생:

- Plateau(정체기): 한참 동안 성능이 안 오르다가 갑자기 툭 떨어지며 좋아지는 경우.

- Learning Rate Scheduler: 학습률이 줄어드는 시점(Step) 이후에야 성능이 폭발하는 경우.

- 불안정한 초기 학습: 초반에 Loss가 심하게 요동치지만 결국 수렴하는 경우.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna

def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    model = nn.Linear(10, 1)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # 가상 데이터
    data, target = torch.randn(32, 10), torch.randn(32, 1)

    for step in range(100):
        optimizer.zero_grad()
        loss = nn.MSELoss()(model(data), target)
        loss.backward()
        optimizer.step()

        # [보고] 현재 성적 보고
        trial.report(loss.item(), step)

        # [가지치기 확인]
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return loss.item()

# PatientPruner 설정
# MedianPruner를 기본으로 하되, 5번의 기회를 더 줍니다.
base_pruner = optuna.pruners.MedianPruner()
pruner = optuna.pruners.PatientPruner(base_pruner, patience=5)

study = optuna.create_study(direction="minimize", pruner=pruner)
study.optimize(objective, n_trials=20)

print(f"최적 파라미터: {study.best_params}")

- Optuna의 가장 강력한 장점 중 하나는 학습 과정을 한눈에 파악할 수 있는 시각화


In [ ]:
%pip install optuna plotly

In [ ]:
import optuna
import pandas as pd
import plotly.express as px
from optuna.visualization import (
    plot_optimization_history,     # 최적화 흐름
    plot_param_importances,       # 파라미터 중요도 (가장 중요)
    plot_parallel_coordinate,     # 파라미터 간의 연관관계 (고득점의 길)
    plot_contour                 # 두 파라미터의 등고선 분석 (Sweet Spot)
)

# 1. 시각화용 가상 Objective 함수 정의
# (실제 학습은 하지 않고, 파라미터 조합에 따른 그럴듯한 점수를 반환합니다)
def objective_visual(trial):
    # 탐색 공간 정의
    n_layers = trial.suggest_int("n_layers", 1, 5)
    hidden_dim = trial.suggest_int("hidden_dim", 32, 256, step=32)
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    optimizer = trial.suggest_categorical("optimizer", ["Adam", "SGD", "RMSprop"])
    dropout = trial.suggest_float("dropout", 0.1, 0.5)

    # 파라미터 간의 복잡한 연관관계를 시뮬레이션하는 가상 점수 계산
    # (예: Adam이면서 hidden_dim이 크고 lr이 적당할 때 고득점)
    base_score = (hidden_dim / 256) * 0.5 + (n_layers * 0.1)

    if optimizer == "Adam":
        opt_bonus = 0.3
        # lr과 hidden_dim의 연관관계 시뮬레이션: Adam은 작은 lr에서 성능 폭발
        lr_factor = 1.0 - abs(lr - 0.001) * 10
    else:
        opt_bonus = 0.1
        lr_factor = 0.5

    # 최종 점수 (최소화 목표)
    score = 1.0 - (base_score + opt_bonus + lr_factor - dropout * 0.2)
    return max(0.01, score) # 음수 방지

# 2. Study 생성 및 최적화 실행 (데이터 수집)
# 50번의 시도를 통해 시각화에 충분한 데이터를 모읍니다.
study = optuna.create_study(direction="minimize")
study.optimize(objective_visual, n_trials=50, show_progress_bar=True)

# 3. 결과 데이터프레임 확인 (선택 사항)
# df = study.trials_dataframe()
# print(df.head())

In [ ]:
# 파라미터 간의 고차원 상호작용을 선으로 연결하여 시각화합니다.
plot_parallel_coordinate(study).show()

In [ ]:
# 성능에 가장 큰 영향을 미친 파라미터를 하이퍼파라미터 중요도로 보여줍니다.
plot_param_importances(study).show()

# W&B(Weights & Biases)

## 1. W&B가 무엇인가

W&B(Weights & Biases)는 머신러닝 실험을 기록하고 비교하는 도구입.  
학습할 때 생기는 **loss, accuracy, learning rate, 하이퍼파라미터, 모델 체크포인트, 그래프, 이미지** 등을 한 곳에 정리해서 볼 수 있게 해줌



- 어떤 설정으로 학습했는지 기록
- epoch마다 성능이 어떻게 변했는지 기록
- 여러 실험(run)을 한 화면에서 비교
- 가장 좋은 실험을 찾기 쉽게 정리
- 모델 파일이나 결과 이미지를 함께 저장

---

## 2. 왜 W&B를 쓰는가

PyTorch로 실험을 하다 보면 이런 문제가 자주 생깁니다.

- 이번 실험의 learning rate가 뭐였는지 기억이 안 남
- 어떤 run이 제일 좋았는지 헷갈림
- loss는 줄었는데 accuracy는 왜 안 올랐는지 비교가 어려움
- 노트북 여러 개, 실험 여러 번 돌리다 보면 기록이 흩어짐

W&B를 쓰면 이런 정보가 **run 단위로 자동 정리**됩니다. 공식 문서에 따르면 `wandb.init()`으로 run을 시작하고, `wandb.log()`로 step별 지표를 기록하면 로컬의 `wandb` 디렉토리에 저장된 뒤 W&B 클라우드 또는 private server로 동기화됨.

---

## 3. 핵심 개념

### 3-1. Run

**Run**은 한 번의 실험.

예를 들어:

- learning rate = 0.001, batch size = 32 로 학습한 실험 1개
- learning rate = 0.01, batch size = 64 로 학습한 실험 1개

이 각각이 하나의 run.

보통 `wandb.init()`을 호출하면 run이 하나 시작됩니다. W&B 문서는 `wandb.init()`이 새 run을 시작하고 기본적으로 실시간 동기화를 수행한다고 설명.

---

### 3-2. Project

**Project**는 여러 run을 묶는 폴더 같은 개념.

예를 들어:

- 프로젝트 이름: `mnist-classification`
- 그 안에 여러 run이 쌓임

같은 문제를 여러 설정으로 반복 실험할 때 project 안에서 비교하게 됩니다. W&B quickstart도 `project` 이름 아래 run을 기록하는 예시를 제공.

---

### 3-3. Config

**Config**는 하이퍼파라미터나 실험 설정.

예:
- epochs
- learning_rate
- batch_size
- optimizer
- hidden_dim

이 값을 `wandb.init(config=...)`에 넣으면, 나중에 대시보드에서 각 run의 설정을 쉽게 비교할 수 있음. 공식 quickstart와 PyTorch integration 문서에서 `config` 딕셔너리 사용을 안내.
---

### 3-4. Log

**Log**는 학습 중간에 metric을 기록하는 것.

예:
- train loss
- validation loss
- validation accuracy
- learning rate

W&B 문서는 `wandb.Run.log()`가 step마다 key-value 형태의 데이터를 기록한다고 설명.

---

## 4. 가장 기본 흐름

W&B의 가장 기본적인 흐름은 아래와 같습니다.

1. 로그인
2. run 시작
3. config 저장
4. epoch마다 metric 기록
5. run 종료

In [ ]:
%pip install wandb

In [ ]:
import wandb

# 복사하신 API Key를 직접 입력합니다.
wandb.login(key="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")

In [ ]:
import torch
import torch.nn as nn
import wandb

wandb.login()

config = {
    "learning_rate": 1e-3,
    "epochs": 10,
    "batch_size": 64
}

run = wandb.init(project="pytorch-demo", config=config)

# 예시용 모델
model = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU(),
    nn.Linear(64, 2)
)

optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
criterion = nn.CrossEntropyLoss()

# 예시용 더미 데이터
X = torch.randn(256, 20)
y = torch.randint(0, 2, (256,))

for epoch in range(config["epochs"]):
    model.train()
    optimizer.zero_grad()

    logits = model(X)
    loss = criterion(logits, y)

    loss.backward()
    optimizer.step()

    pred = torch.argmax(logits, dim=1)
    acc = (pred == y).float().mean().item()

    wandb.log({
        "epoch": epoch,
        "train_loss": loss.item(),
        "train_acc": acc
    })

wandb.finish()